# Second-Order Taylor Approximation

Chapter 11: Matrix Calculus · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare exact function values with local second-order Taylor expansion around a chosen point.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-11/taylor-second-order/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's function and defaults: f(x) = sin x + x^2 / 4, center x0 = 0.5, span 1.5
x0, span = 0.5, 1.5


def f(x):
    return torch.sin(x) + 0.25 * x ** 2


# ── f'(x0) and f''(x0) by autograd ──────────────────────────────────────────
x = torch.tensor(x0, dtype=torch.float64, requires_grad=True)
(d1,) = torch.autograd.grad(f(x), x, create_graph=True)
(d2,) = torch.autograd.grad(d1, x)
print(f"f(x0) = {f(x).item():.6f}, f'(x0) = {d1.item():.6f}, f''(x0) = {d2.item():.6f}")
print(f"by hand: f' = cos x0 + x0/2 = {torch.cos(torch.tensor(x0)).item() + x0 / 2:.6f}, "
      f"f'' = -sin x0 + 1/2 = {-torch.sin(torch.tensor(x0)).item() + 0.5:.6f}")


def taylor(xs, order):
    dx = xs - x0
    out = f(torch.tensor(x0, dtype=torch.float64)) + d1.detach() * dx
    return out + 0.5 * d2 * dx ** 2 if order == 2 else out


xs = torch.linspace(x0 - span, x0 + span, 321, dtype=torch.float64)
print(f"\nmax error over [x0-{span}, x0+{span}]: first order {(f(xs) - taylor(xs, 1)).abs().max().item():.4f}, "
      f"second order {(f(xs) - taylor(xs, 2)).abs().max().item():.4f}")

# ── The second-order error shrinks like h^3 ─────────────────────────────────
# (f''(x0) is only 0.02 here, so the first-order error is unusually small too;
#  what matters is the rate: order-2 error / h^3 settles to a constant.)
print("\n   h        error (order 1)   error (order 2)   order-2 error / h^3")
for h in (0.4, 0.2, 0.1, 0.05):
    pt = torch.tensor(x0 + h, dtype=torch.float64)
    e1 = (f(pt) - taylor(pt, 1)).abs().item()
    e2 = (f(pt) - taylor(pt, 2)).abs().item()
    print(f"{h:5.2f}   {e1:.3e}         {e2:.3e}         {e2 / h ** 3:.4f}")
print("limit of error/h^3 = |f'''(x0)| / 6 = |cos x0| / 6 =", abs(torch.cos(torch.tensor(x0)).item()) / 6)

f(x0) = 0.541926, f'(x0) = 1.127583, f''(x0) = 0.020574
by hand: f' = cos x0 + x0/2 = 1.127583, f'' = -sin x0 + 1/2 = 0.020574

max error over [x0-1.5, x0+1.5]: first order 0.5580, second order 0.5348

   h        error (order 1)   error (order 2)   order-2 error / h^3
 0.40   7.132e-03         8.778e-03         0.1372
 0.20   7.244e-04         1.136e-03         0.1420
 0.10   4.132e-05         1.442e-04         0.1442
 0.05   7.562e-06         1.816e-05         0.1452
limit of error/h^3 = |f'''(x0)| / 6 = |cos x0| / 6 = 0.14626375834147134
